In [7]:
!unzip mnist_data.zip

Archive:  mnist_data.zip
   creating: mnist_data/
  inflating: mnist_data/quant_params.hex  
  inflating: mnist_data/expected_class.hex  
  inflating: mnist_data/fc2_bias.hex  
  inflating: mnist_data/fc1_output.hex  
 extracting: mnist_data/fc2_output.hex  
  inflating: mnist_data/input_image.hex  
  inflating: mnist_data/fc1_bias.hex  
  inflating: mnist_data/fc2_weight_tiles.hex  
  inflating: mnist_data/fc1_weight_tiles.hex  


In [8]:
from pynq import Overlay, MMIO
import numpy as np
import os

In [9]:
# ====================================================================
# 1. 加载 bitstream
# ====================================================================
ol = Overlay("cim_soc.bit")
print("Overlay loaded!")

BASE = 0x40000000
mmio = MMIO(BASE, 0x4000)   # 16KB address space

Overlay loaded!


In [10]:
# ====================================================================
# 2. CSR 地址定义 (与 cim_pkg.sv 完全对应, 14-bit addr space)
# ====================================================================
CTRL          = 0x000
STATUS        = 0x004
CSR_IN_DIM    = 0x010
CSR_OUT_DIM   = 0x014
CSR_N_IB      = 0x018
CSR_N_OB      = 0x01C
REQUANT_MULT  = 0x020
REQUANT_SHIFT = 0x024
INPUT_ZP      = 0x028
ACT_MODE      = 0x02C
CYCLE_CNT_LO  = 0x030
MAC_CNT_LO    = 0x038
PRED_CLASS    = 0x040
WDMA_ADDR     = 0x044
WDMA_DATA     = 0x048
WDMA_CTRL     = 0x04C
LOGIT_BASE    = 0x100
MEM_INPUT     = 0x1000
MEM_BIAS      = 0x2000

In [11]:
# ====================================================================
# 3. 内嵌 Golden Model (不依赖外部 hex 文件)
# ====================================================================
TILE_ROWS = 16
TILE_COLS = 16
ELEMS_PER_CHUNK = 4   # 32 / 8
CHUNKS_PER_ROW  = 4   # TILE_COLS / ELEMS_PER_CHUNK

def apply_zero_point(x_uint8, zero_point):
    x_eff = x_uint8.astype(np.int32) - zero_point
    return np.clip(x_eff, 0, 511).astype(np.uint16)

def requantize_int32_to_int8(x, mult, rshift):
    result = np.zeros(len(x), dtype=np.int8)
    for i in range(len(x)):
        prod = int(x[i]) * int(mult)
        if rshift == 0:
            shifted = prod
        else:
            shifted = (prod + (1 << (rshift - 1))) >> rshift
        shifted = max(-128, min(127, shifted))
        result[i] = np.int8(shifted)
    return result

def infer_layer(input_uint8, weight_int8, bias_int32,
                zero_point=-128, requant_mult=1, requant_shift=0, activation="relu"):
    x_eff = apply_zero_point(input_uint8, zero_point)
    acc = weight_int8.astype(np.int32) @ x_eff.astype(np.int32)
    acc_bias = acc + bias_int32.astype(np.int32)
    if activation == "relu":
        activated = np.maximum(acc_bias, 0)
    else:
        activated = acc_bias
    output = requantize_int32_to_int8(activated, requant_mult, requant_shift)
    return output

def calibrate_requant(acc_values, shift=16):
    max_abs = max(abs(int(acc_values.max())), abs(int(acc_values.min())), 1)
    scale = 127.0 / max_abs
    mult = int(round(scale * (1 << shift)))
    return max(1, mult), shift

def weight_to_chunks(weight_int8):
    out_dim, in_dim = weight_int8.shape
    n_ob = (out_dim + TILE_ROWS - 1) // TILE_ROWS
    n_ib = (in_dim + TILE_COLS - 1) // TILE_COLS
    chunks = []
    for ob in range(n_ob):
        for ib in range(n_ib):
            for chunk in range(TILE_ROWS * CHUNKS_PER_ROW):
                row = chunk // CHUNKS_PER_ROW
                col_group = chunk % CHUNKS_PER_ROW
                word = 0
                for b in range(ELEMS_PER_CHUNK):
                    oi = ob * TILE_ROWS + row
                    ii = ib * TILE_COLS + col_group * ELEMS_PER_CHUNK + b
                    if oi < out_dim and ii < in_dim:
                        val = int(weight_int8[oi, ii]) & 0xFF
                    else:
                        val = 0
                    word |= val << (b * 8)
                chunks.append(word)
    return chunks

def bias_to_u32(bias_int32):
    return [int(b) & 0xFFFFFFFF for b in bias_int32]

print("Golden model functions loaded.")

Golden model functions loaded.


In [12]:
# ====================================================================
# 4. 生成 Golden 数据 (与 golden_model.py --mnist-e2e --seed 42 完全一致)
# ====================================================================
np.random.seed(42)
w1 = np.random.randint(-128, 127, (128, 784), dtype=np.int8)
b1 = np.random.randint(-5000, 5000, 128, dtype=np.int32)
w2 = np.random.randint(-128, 127, (10, 128), dtype=np.int8)
b2 = np.random.randint(-5000, 5000, 10, dtype=np.int32)
img = np.random.randint(0, 255, 784, dtype=np.uint8)

# Calibrate FC1
x_eff1 = np.clip(img.astype(np.int32) - (-128), 0, 511).astype(np.int32)
acc1 = w1.astype(np.int32) @ x_eff1 + b1.astype(np.int32)
relu1 = np.maximum(acc1, 0)
fc1_mult, fc1_shift = calibrate_requant(relu1, shift=16)

# Run FC1 golden
fc1_golden = infer_layer(img, w1, b1, -128, fc1_mult, fc1_shift, "relu")

# Calibrate FC2
fc2_in = fc1_golden.view(np.uint8)
x_eff2 = np.clip(fc2_in.astype(np.int32) - 0, 0, 511).astype(np.int32)
acc2 = w2.astype(np.int32) @ x_eff2 + b2.astype(np.int32)
fc2_mult, fc2_shift = calibrate_requant(acc2, shift=16)

# Run FC2 golden
fc2_golden = infer_layer(fc2_in, w2, b2, 0, fc2_mult, fc2_shift, "none")
expected_class = int(np.argmax(fc2_golden))

# Pack data for hardware
fc1_weight_chunks = weight_to_chunks(w1)
fc2_weight_chunks = weight_to_chunks(w2)
fc1_bias_u32      = bias_to_u32(b1)
fc2_bias_u32      = bias_to_u32(b2)
input_image       = img.tolist()

print(f"FC1 weights: {len(fc1_weight_chunks)} chunks")
print(f"FC2 weights: {len(fc2_weight_chunks)} chunks")
print(f"Quant: fc1_mult={fc1_mult}, fc1_shift={fc1_shift}")
print(f"       fc2_mult={fc2_mult}, fc2_shift={fc2_shift}")
print(f"FC1 golden output (first 10): {fc1_golden[:10]}")
print(f"FC2 golden output: {fc2_golden}")
print(f"Expected class: {expected_class}")

FC1 weights: 25088 chunks
FC2 weights: 512 chunks
Quant: fc1_mult=10, fc1_shift=16
       fc2_mult=118, fc2_shift=16
FC1 golden output (first 10): [ 0  1  4  0 84 85  0  0 10  0]
FC2 golden output: [  61  -33   98 -100  -43   26  -50  -86  127    6]
Expected class: 8


In [13]:
# ====================================================================
# 4b. (可选) 对比 hex 文件数据与内嵌 golden model 的结果
# ====================================================================
DATA_DIR = "mnist_data"
try:
    def read_hex_u8(fn):
        with open(os.path.join(DATA_DIR, fn)) as f:
            return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

    hex_fc1 = np.array(read_hex_u8("fc1_output.hex"), dtype=np.uint8).view(np.int8)
    hex_fc2 = np.array(read_hex_u8("fc2_output.hex"), dtype=np.uint8).view(np.int8)

    fc1_hex_match = np.array_equal(fc1_golden, hex_fc1)
    fc2_hex_match = np.array_equal(fc2_golden, hex_fc2)
    print(f"Hex vs inline golden - FC1: {'MATCH ✓' if fc1_hex_match else 'MISMATCH ✗'}")
    print(f"Hex vs inline golden - FC2: {'MATCH ✓' if fc2_hex_match else 'MISMATCH ✗'}")
    if not fc1_hex_match:
        diffs = np.where(fc1_golden != hex_fc1)[0]
        print(f"  FC1 differs at {len(diffs)} indices: {diffs[:10].tolist()}...")
        for d in diffs[:5]:
            print(f"    [{d}] inline={fc1_golden[d]}, hex={hex_fc1[d]}")
    if not fc2_hex_match:
        diffs = np.where(fc2_golden != hex_fc2)[0]
        print(f"  FC2 differs at {len(diffs)} indices:")
        for d in diffs:
            print(f"    [{d}] inline={fc2_golden[d]}, hex={hex_fc2[d]}")
except Exception as e:
    print(f"(无法读取 hex 文件: {e} — 跳过对比)")

Hex vs inline golden - FC1: MATCH ✓
Hex vs inline golden - FC2: MATCH ✓


In [14]:
# ====================================================================
# 5. 硬件操作工具函数
# ====================================================================
def soft_reset():
    mmio.write(CTRL, 0x4)

def clear_done():
    mmio.write(CTRL, 0x2)

def configure_layer(in_dim, out_dim, zp, mult, shift, act, verify=True):
    """配置一层的 CSR 参数, 可选回读验证"""
    n_ib = (in_dim + 15) // 16
    n_ob = (out_dim + 15) // 16
    writes = [
        ("CSR_IN_DIM",    CSR_IN_DIM,    in_dim),
        ("CSR_OUT_DIM",   CSR_OUT_DIM,   out_dim),
        ("CSR_N_IB",      CSR_N_IB,      n_ib),
        ("CSR_N_OB",      CSR_N_OB,      n_ob),
        ("REQUANT_MULT",  REQUANT_MULT,  mult),
        ("REQUANT_SHIFT", REQUANT_SHIFT, shift),
        ("INPUT_ZP",      INPUT_ZP,      zp & 0xFFFFFFFF),
        ("ACT_MODE",      ACT_MODE,      act),
    ]
    for name, addr, val in writes:
        mmio.write(addr, val & 0xFFFFFFFF)
    if verify:
        for name, addr, val in writes:
            rb = mmio.read(addr)
            expected = val & 0xFFFFFFFF
            if rb != expected:
                print(f"  ✗ CSR MISMATCH: {name} wrote=0x{expected:08x} read=0x{rb:08x}")

def load_weights_burst(chunks):
    mmio.write(WDMA_ADDR, 0)
    mmio.write(WDMA_CTRL, 0x02)
    for c in chunks:
        mmio.write(WDMA_DATA, int(c))
    mmio.write(WDMA_CTRL, 0x00)

def load_bias(bias_list):
    for i, b in enumerate(bias_list):
        mmio.write(MEM_BIAS + 4*i, int(b) & 0xFFFFFFFF)

def load_input(data_u8):
    padded = list(data_u8)
    while len(padded) % 16 != 0:
        padded.append(0)
    for i, x in enumerate(padded):
        mmio.write(MEM_INPUT + 4*i, int(x) & 0xFF)

def run_inference():
    clear_done()
    mmio.write(CTRL, 0x1)
    while not (mmio.read(STATUS) & 0x2):
        pass
    cycles = mmio.read(CYCLE_CNT_LO)
    macs   = mmio.read(MAC_CNT_LO)
    return cycles, macs

def read_output(out_dim):
    out = []
    for i in range(out_dim):
        v = mmio.read(LOGIT_BASE + 4*i)
        out.append(np.uint8(v & 0xFF).view(np.int8))
    return out

In [15]:
# ====================================================================
# 6. 连通性测试
# ====================================================================
print("\n=== AXI Connectivity Test ===")
soft_reset()
status = mmio.read(STATUS)
print(f"  STATUS after reset: 0x{status:08x}")

mmio.write(CSR_IN_DIM, 784)
readback = mmio.read(CSR_IN_DIM)
print(f"  IN_DIM write=784, readback={readback}  {'PASS' if readback == 784 else 'FAIL'}")

if readback != 784:
    print("ERROR: AXI read/write failed! Check address mapping.")
    raise RuntimeError("AXI connectivity test failed")


=== AXI Connectivity Test ===
  STATUS after reset: 0x00000000
  IN_DIM write=784, readback=784  PASS


In [16]:
# ====================================================================
# 6b. CSR 回读诊断 + 时序违例检测
# ====================================================================
# !! 你的 bitstream 有严重的时序违例 (WNS = -20.727ns) !!
# !! 这是所有计算错误的根本原因 !!
#
# 最差路径: row_idx_reg → 44级逻辑 → argmax (需要28.7ns, 时钟只有8ns)
# 关键瓶颈: cim_tile.sv 的16元素乘加链 + activation_unit + requantize
#           全部是组合逻辑, 路径太长无法在125MHz下收敛
#
# 修复方案 (任选其一):
#   1. 降频: vivado_build.tcl 中 FCLK_MHZ 从 125 改为 50 (或更低)
#   2. 流水线化 cim_tile: 把16列乘加拆成2-4级pipeline
#   3. 流水线化 activation_unit: requantize 拆成独立的流水级
#
# 以下测试验证 CSR 寄存器是否可读写 (不受时序违例影响):
print("\n=== CSR Readback Diagnostic ===")

test_cases = [
    ("CSR_IN_DIM",    CSR_IN_DIM,    784),
    ("CSR_OUT_DIM",   CSR_OUT_DIM,   128),
    ("CSR_N_IB",      CSR_N_IB,      49),
    ("CSR_N_OB",      CSR_N_OB,      8),
    ("REQUANT_MULT",  REQUANT_MULT,  10),
    ("REQUANT_SHIFT", REQUANT_SHIFT, 16),
    ("ACT_MODE",      ACT_MODE,      1),
]

csr_errors = 0
for name, addr, val in test_cases:
    mmio.write(addr, val & 0xFFFFFFFF)
    rb = mmio.read(addr)
    ok = "PASS" if rb == val else "FAIL"
    if rb != val:
        csr_errors += 1
        print(f"  {name:16s}: wrote={val}, readback={rb}  {ok} ✗")
    else:
        print(f"  {name:16s}: wrote={val}, readback={rb}  {ok}")

# Test INPUT_ZP (signed value)
mmio.write(INPUT_ZP, (-128) & 0xFFFFFFFF)
rb_zp = mmio.read(INPUT_ZP)
expected_zp = (-128) & 0xFFFFFFFF  # 0xFFFFFF80
ok_zp = "PASS" if rb_zp == expected_zp else "FAIL"
if rb_zp != expected_zp:
    csr_errors += 1
print(f"  {'INPUT_ZP':16s}: wrote=0x{expected_zp:08x}, readback=0x{rb_zp:08x}  {ok_zp}")

if csr_errors == 0:
    print("  CSR registers: ALL OK ✓")
    print("  (CSR 寄存器没问题, 计算错误来自 cim_tile/activation 的时序违例)")
else:
    print(f"  CSR registers: {csr_errors} FAILURES ✗")

print()
print("⚠️  TIMING VIOLATION DETECTED IN BITSTREAM")
print("   WNS = -20.727ns (需要 28.7ns, 时钟周期 8ns)")
print("   2914 failing endpoints — 计算结果不可信")
print()
print("🔧 快速修复: 把 FCLK_MHZ 从 125 降到 25-50, 重新综合")
print("   在 vivado_build.tcl 第21行修改:")
print("   set FCLK_MHZ  50    ;# 或者 25")
print()
print("🔧 正确修复: 给 cim_tile 加流水线")
print("   把 cim_tile.sv 的16列组合乘加拆成多级寄存器")


=== CSR Readback Diagnostic ===
  CSR_IN_DIM      : wrote=784, readback=784  PASS
  CSR_OUT_DIM     : wrote=128, readback=128  PASS
  CSR_N_IB        : wrote=49, readback=49  PASS
  CSR_N_OB        : wrote=8, readback=8  PASS
  REQUANT_MULT    : wrote=10, readback=10  PASS
  REQUANT_SHIFT   : wrote=16, readback=16  PASS
  ACT_MODE        : wrote=1, readback=1  PASS
  INPUT_ZP        : wrote=0xffffff80, readback=0xffffff80  PASS
  CSR registers: ALL OK ✓
  (CSR 寄存器没问题, 计算错误来自 cim_tile/activation 的时序违例)

⚠️  TIMING VIOLATION DETECTED IN BITSTREAM
   WNS = -20.727ns (需要 28.7ns, 时钟周期 8ns)
   2914 failing endpoints — 计算结果不可信

🔧 快速修复: 把 FCLK_MHZ 从 125 降到 25-50, 重新综合
   在 vivado_build.tcl 第21行修改:
   set FCLK_MHZ  50    ;# 或者 25

🔧 正确修复: 给 cim_tile 加流水线
   把 cim_tile.sv 的16列组合乘加拆成多级寄存器


In [17]:
# ====================================================================
# 7. FC1: 784 → 128, ReLU
# ====================================================================
print("\n=== Layer 1: FC1 784→128 ReLU ===")
soft_reset()

configure_layer(in_dim=784, out_dim=128, zp=-128, mult=fc1_mult, shift=fc1_shift, act=1)
print("  Loading weights...")
load_weights_burst(fc1_weight_chunks)
print("  Loading bias...")
load_bias(fc1_bias_u32)
print("  Loading input...")
load_input(input_image)

print("  Running inference...")
cycles, macs = run_inference()
print(f"  Done! cycles={cycles}, MACs={macs}")

fc1_out = read_output(128)

# 对比 FC1
fc1_err = 0
for i in range(128):
    if fc1_out[i] != fc1_golden[i]:
        print(f"  FC1 MISMATCH [{i}]: HW={fc1_out[i]}, Golden={fc1_golden[i]}")
        fc1_err += 1

if fc1_err == 0:
    print(f"  FC1: ALL 128 outputs MATCH ✓")
else:
    print(f"  FC1: {fc1_err}/128 MISMATCHES ✗")


=== Layer 1: FC1 784→128 ReLU ===
  Loading weights...
  Loading bias...
  Loading input...
  Running inference...
  Done! cycles=1968, MACs=100352
  FC1: ALL 128 outputs MATCH ✓


In [18]:
# ====================================================================
# 8. FC2: 128 → 10, no activation
# ====================================================================
print("\n=== Layer 2: FC2 128→10 (no activation) ===")

configure_layer(in_dim=128, out_dim=10, zp=0, mult=fc2_mult, shift=fc2_shift, act=0)
print("  Loading weights...")
load_weights_burst(fc2_weight_chunks)
print("  Loading bias...")
load_bias(fc2_bias_u32)
print("  Loading FC1 output as input...")
fc1_out_u8 = [int(x) & 0xFF for x in fc1_out]
load_input(fc1_out_u8)

print("  Running inference...")
cycles, macs = run_inference()
print(f"  Done! cycles={cycles}, MACs={macs}")

fc2_out = read_output(10)

# 对比 FC2
fc2_err = 0
for i in range(10):
    status_str = "OK" if fc2_out[i] == fc2_golden[i] else "MISMATCH"
    print(f"  logit[{i}] = {fc2_out[i]:4d}  (golden={fc2_golden[i]:4d})  {status_str}")
    if fc2_out[i] != fc2_golden[i]:
        fc2_err += 1

if fc2_err == 0:
    print(f"  FC2: ALL 10 outputs MATCH ✓")
else:
    print(f"  FC2: {fc2_err}/10 MISMATCHES ✗")


=== Layer 2: FC2 128→10 (no activation) ===
  Loading weights...
  Loading bias...
  Loading FC1 output as input...
  Running inference...
  Done! cycles=82, MACs=2048
  logit[0] =   61  (golden=  61)  OK
  logit[1] =  -33  (golden= -33)  OK
  logit[2] =   98  (golden=  98)  OK
  logit[3] = -100  (golden=-100)  OK
  logit[4] =  -43  (golden= -43)  OK
  logit[5] =   26  (golden=  26)  OK
  logit[6] =  -50  (golden= -50)  OK
  logit[7] =  -86  (golden= -86)  OK
  logit[8] =  127  (golden= 127)  OK
  logit[9] =    6  (golden=   6)  OK
  FC2: ALL 10 outputs MATCH ✓


In [19]:
# ====================================================================
# 9. Argmax 检查 + 总结
# ====================================================================
hw_pred = mmio.read(PRED_CLASS)
print(f"\n=== Result ===")
print(f"  HW predicted class: {hw_pred}")
print(f"  Golden expected:    {expected_class}")
print(f"  Argmax: {'MATCH ✓' if hw_pred == expected_class else 'MISMATCH ✗'}")

total_err = fc1_err + fc2_err + (0 if hw_pred == expected_class else 1)
print(f"\n{'='*60}")
if total_err == 0:
    print(">>> ALL ON-BOARD TESTS PASSED <<<")
else:
    print(f">>> {total_err} ERRORS DETECTED <<<")
print(f"{'='*60}")


=== Result ===
  HW predicted class: 8
  Golden expected:    8
  Argmax: MATCH ✓

>>> ALL ON-BOARD TESTS PASSED <<<
